In [8]:
import pandas as pd
from rapidfuzz.fuzz import ratio
from rapidfuzz.distance import DamerauLevenshtein

base_codes = [
    "8056E4","2265K","8813B","C1234","L4567",
    "P7890","A1111","D4321","F6789","X5555",
    "M3344","Q9090","T8080","R2244","Z7777",
    "H1200","K8888","B4561","N9091","Y6543",
    "V3322","G1000","S7654","U2468","J1357"
]

pairs = []

# OCR-like substitutions
ocr_map = {
    "0":"O", "1":"I", "2":"Z", "5":"S",
    "6":"G", "8":"B", "B":"8", "G":"6",
    "I":"1", "O":"0", "S":"5", "Z":"2"
}

for code in base_codes:
    for i, ch in enumerate(code):
        if ch in ocr_map:
            corrupted = code[:i] + ocr_map[ch] + code[i+1:]
            pairs.append((code, corrupted))

# Transpositions
for code in base_codes:
    for i in range(len(code)-1):
        swapped = list(code)
        swapped[i], swapped[i+1] = swapped[i+1], swapped[i]
        pairs.append((code, "".join(swapped)))

# Insertions
for code in base_codes:
    pairs.append((code, code + "+"))
    pairs.append((code, code + "-"))
    pairs.append((code, code + "."))
    pairs.append((code, code[:3] + "-" + code[3:]))

# Deletions
for code in base_codes:
    pairs.append((code, code[:-1]))
    pairs.append((code, code[1:]))

# Moderate modifications
for code in base_codes:
    pairs.append((code, code[::-1]))
    pairs.append((code, code[-1] + code[:-1]))
    pairs.append((code, code[1:] + code[0]))


results = []

for a, b in pairs:

    ratio_score = round(ratio(a, b), 2)

    damerau_score = round(
        DamerauLevenshtein.normalized_similarity(a, b) * 100,
        2
    )

    results.append([
        a,
        b,
        ratio_score,
        damerau_score
    ])

df = pd.DataFrame(results, columns=[
    "string_A",
    "string_B",
    "ratio_score",
    "damerau_score"
])

df.to_csv("comparison_scores.csv", index=False)

print(df.head(20))

   string_A string_B  ratio_score  damerau_score
0    8056E4   B056E4        83.33          83.33
1    8056E4   8O56E4        83.33          83.33
2    8056E4   80S6E4        83.33          83.33
3    8056E4   805GE4        83.33          83.33
4     2265K    Z265K        80.00          80.00
5     2265K    2Z65K        80.00          80.00
6     2265K    22G5K        80.00          80.00
7     2265K    226SK        80.00          80.00
8     8813B    B813B        80.00          80.00
9     8813B    8B13B        80.00          80.00
10    8813B    88I3B        80.00          80.00
11    8813B    88138        80.00          80.00
12    C1234    CI234        80.00          80.00
13    C1234    C1Z34        80.00          80.00
14    L4567    L4S67        80.00          80.00
15    L4567    L45G7        80.00          80.00
16    P7890    P7B90        80.00          80.00
17    P7890    P789O        80.00          80.00
18    A1111    AI111        80.00          80.00
19    A1111    A1I11